# Phase 4A — deterministic training calibration

This notebook is a thin Colab orchestrator. It reads targets only from the frozen `train` split, verifies that both feature profiles are aligned, runs the full test suite, and writes the reproducible `calibration_manifest.json` to Drive. It does not train a model or access validation/test labels.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_ROOT = Path('/content/temporalgnn-nids')
REPO_URL = 'https://github.com/tatipar/temporalgnn-nids.git'
BRANCH = 'feat/fair-retrain-clean'

if REPO_ROOT.exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'switch', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)],
        check=True,
    )

sys.path.insert(0, str(REPO_ROOT / 'code/python'))

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric'], check=True)

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
GRAPH_VERSION = 'infiltration_v1_w30_tcpflags_episode_split_v1'
GRAPH_ROOT = Path('/content/drive/MyDrive/nids-fair-retrain/graphs') / GRAPH_VERSION
CALIBRATION_ROOT = Path('/content/drive/MyDrive/nids-fair-retrain/calibration') / GRAPH_VERSION
CALIBRATION_MANIFEST = CALIBRATION_ROOT / 'calibration_manifest.json'

# Set only if the corrected-data manifest was moved after graph construction.
CORRECTED_MANIFEST_OVERRIDE = None
VERIFY_GRAPH_CHECKSUMS = False  # Enable after copying/moving the graph collection.
OVERWRITE_DIFFERENT_MANIFEST = False  # Identical reruns never require overwrite.

assert (GRAPH_ROOT / 'graph_manifest.json').is_file(), GRAPH_ROOT
print({'graph_root': str(GRAPH_ROOT), 'output': str(CALIBRATION_MANIFEST)})

## Acceptance tests

Run the complete suite against the exact checked-out revision before publishing the calibration artifact.

In [ ]:
tests_root = REPO_ROOT / 'code/python/tests'
test_env = dict(__import__('os').environ)
test_env['PYTHONPATH'] = str(REPO_ROOT / 'code/python')
subprocess.run(
    [
        sys.executable, '-m', 'unittest', 'discover',
        '-s', str(tests_root), '-p', 'test_*.py', '-v',
    ],
    cwd=REPO_ROOT,
    env=test_env,
    check=True,
)

## Generate the frozen candidate grid

The CLI refuses to replace a different existing artifact unless the overwrite flag is deliberately enabled. An identical rerun reports `unchanged`.

In [ ]:
command = [
    sys.executable,
    str(REPO_ROOT / 'code/python/scripts/calibrate_pos_weight.py'),
    '--graph-root', str(GRAPH_ROOT),
    '--output', str(CALIBRATION_MANIFEST),
]
if CORRECTED_MANIFEST_OVERRIDE is not None:
    command.extend(['--corrected-manifest', str(CORRECTED_MANIFEST_OVERRIDE)])
if VERIFY_GRAPH_CHECKSUMS:
    command.append('--verify-checksums')
if OVERWRITE_DIFFERENT_MANIFEST:
    command.append('--overwrite')

subprocess.run(command, cwd=REPO_ROOT, check=True)

In [ ]:
import hashlib
import json

serialized = CALIBRATION_MANIFEST.read_bytes()
manifest = json.loads(serialized)
print('calibration_manifest_sha256:', hashlib.sha256(serialized).hexdigest())
print(json.dumps({
    'counts': manifest['counts'],
    'profile_alignment': manifest['profile_alignment'],
    'candidates': manifest['candidates'],
    'code_revision': manifest['code_revision'],
}, indent=2, sort_keys=True))